#  ongoing orders without paid invoices in the current fiscal year 

In [ ]:
import pandas as pd
import requests
from datetime import datetime

pd.set_option('display.max_columns', None)

## 1. Log in from a shared notebook

This is the notebook that will authenticate you to the FOLIO API and store your access token in a variable called ACCESS_TOKEN. You will need to run this cell before running any other cells in this notebook.

In [ ]:
%run folio_auth.ipynb

## Logic
- Get all POs with a status of open and orderType of Ongoing
- Get all POLs with receiptStatus of Ongoing
- Get all Invoices with an Invoice Date of the current FY
- Merge and highlight those POLs with no invoice date or invoice paid date for the current FY. --> This could capture invoices created, but not approved/paid.


In [ ]:
def fetch_all_records(endpoint, records_key, limit=1000, query=None):
    all_records = []
    offset = 0
    base_url = OKAPI_URL
    headers = HEADERS.copy() if 'HEADERS' in globals() else {"X-Okapi-Tenant": TENANT, "Content-Type": "application/json"}
    if 'token' in globals():
        headers["Authorization"] = f"Bearer {token}"

    while True:
        params = {"limit": limit, "offset": offset}
        if query:
            params["query"] = query
        response = session.get(f"{base_url}{endpoint}", headers=headers, params=params)
        response.raise_for_status()
        payload = response.json()

        batch = payload.get(records_key, [])
        all_records.extend(batch)

        if len(batch) < limit:
            break
        offset += limit
    return all_records

In [ ]:
fys_raw = fetch_all_records(
    "/finance/fiscal-years",
    records_key="fiscalYears"
) 
fys_df = pd.DataFrame(fys_raw)
today = datetime.today().strftime("%Y-%m-%d")
cur_fy_df = fys_df[(fys_df['periodStart'] <= today) & (fys_df['periodEnd'] >= today)]
styled_fy = cur_fy_df.style.set_properties(
  **{'border': '1px solid black', 'background-color': 'lightgrey'}  
)
print(f"Fiscal Years that encompass {today}")
cur_fy_df[
    ['id', 'name', 'periodStart', 'periodEnd']
].style.set_properties(
    **{'border': '1px solid black'}
)

1. Get all POs ongoing POs with a status of open and orderType of Ongoing
2. Get all POLs with receiptStatus of Ongoing
3. Get all Invoices with an Invoice Date of the current FY
4. Merge and highlight rows with no invoices

In [44]:
orders_raw = fetch_all_records(
    "/orders/composite-orders",
    records_key="purchaseOrders",
    query='workflowStatus="Open" and orderType = "Ongoing"'
    ) 
orders_df = pd.DataFrame(orders_raw)

print(f"{len(orders_raw)} open, ongoing orders found")
orders_df.head()

43 open, ongoing orders found


,id,approved,approvedById,approvalDate,billTo,dateOrdered,manualPo,notes,poNumber,orderType,reEncumber,ongoing,shipTo,template,vendor,workflowStatus,acqUnitIds,tags,metadata,customFields,assignedTo
0,40aaa902-bfc8-4f20-af9f-e1cd729acde9,False,4a390634-f254-4777-8448-6cd7a5ad29e3,2025-04-21T17:04:42.479+00:00,789ab62a-9efa-4c70-a6ef-a3615fe4b344,2025-04-21T17:04:42.479+00:00,False,[],21624,Ongoing,True,"{'interval': 365, 'isSubscription': True, 'man...",4d098cbe-04cd-4332-9fad-24208fae34c4,74b9551e-f7e5-43e8-887f-7d6b075e9b19,ae62585b-5585-41a7-9519-5b16c1d32d54,Open,[a3f803ba-587c-4aac-b2fc-35226659c4fe],{'tagList': []},{'createdDate': '2025-04-21T16:59:34.873+00:00...,NaN,NaN
1,180bcb30-7b70-4536-bd0b-6b6cf231ddfa,False,4a390634-f254-4777-8448-6cd7a5ad29e3,2025-05-07T04:31:04.111+00:00,789ab62a-9efa-4c70-a6ef-a3615fe4b344,2025-05-07T04:31:04.111+00:00,False,[],21655,Ongoing,False,"{'interval': 365, 'isSubscription': True, 'man...",NaN,5b0bf4ea-7971-4717-a77c-8ce7f2577685,fed3982c-1e92-4719-bb9b-42129018fd25,Open,[],{'tagList': []},{'createdDate': '2025-05-07T04:29:23.084+00:00...,NaN,NaN
2,e02be3d3-63b4-45a8-a82d-d8e001adb813,False,7c2b76f7-634e-47d1-9aa5-9910751f23c8,2025-05-09T11:10:06.741+00:00,789ab62a-9efa-4c70-a6ef-a3615fe4b344,2025-05-09T11:10:06.741+00:00,NaN,[],21660,Ongoing,True,"{'interval': 1, 'isSubscription': True, 'manua...",4d098cbe-04cd-4332-9fad-24208fae34c4,NaN,1dc0f2dc-91b1-41bb-bb7f-145815385366,Open,[],NaN,{'createdDate': '2025-05-09T11:06:25.857+00:00...,NaN,NaN
3,f74c9128-0666-43c9-8698-0eeea8dfc036,False,91d619a5-66b5-45f3-8680-42525ec90bcd,2025-05-29T12:00:20.752+00:00,NaN,2025-05-29T12:00:20.752+00:00,False,[],21712,Ongoing,False,"{'interval': 12, 'isSubscription': True, 'manu...",NaN,79d99241-f68e-474e-a104-da82219edb9e,1ce90497-a8ba-4940-9ffc-0044793e91a0,Open,[5f3bbae0-5d37-460f-a47f-71224c2952a5],{'tagList': []},{'createdDate': '2025-05-29T11:57:43.165+00:00...,NaN,NaN
4,4fa80b0a-1934-4f90-ba1e-8dcaf208ec29,False,4a390634-f254-4777-8448-6cd7a5ad29e3,2025-06-12T19:59:04.021+00:00,NaN,2025-06-12T19:59:04.021+00:00,False,[],21738,Ongoing,False,"{'isSubscription': False, 'manualRenewal': False}",NaN,15b8c257-7438-4937-9630-d57d2b679cb4,a431ca52-328c-4a97-880a-3689841faba6,Open,[],{'tagList': []},{'createdDate': '2025-06-12T17:18:41.319+00:00...,NaN,NaN


In [45]:
poLines_raw = fetch_all_records(
    "/orders/order-lines",
    records_key="poLines",
    query='receiptStatus = "Ongoing"'
    ) 
poLines_df = pd.DataFrame(poLines_raw)

print(f"{len(poLines_raw)} purchase order lines found with a receipt stats of ongoing")
poLines_df.head()

50 purchase order lines found with a receipt stats of ongoing


,id,edition,checkinItems,acquisitionMethod,automaticExport,alerts,claims,claimingActive,claimingInterval,collection,contributors,cost,details,donorOrganizationIds,fundDistribution,instanceId,isPackage,locations,searchLocationIds,orderFormat,paymentStatus,physical,poLineNumber,publicationDate,publisher,purchaseOrderId,receiptStatus,reportingCodes,rush,source,titleOrPackage,vendorDetail,metadata,eresource,requester,cancellationRestriction,cancellationRestrictionNote,description,receiptDate,renewalNote,customFields
0,2de67573-9f06-4819-8c2d-79aa1da01a70,,True,28f6e038-52d2-4a04-a14b-8dffedd1d8bb,False,[],[],True,30.0,False,[],"{'listUnitPrice': 300.0, 'currency': 'USD', 'd...",{'receivingNote': 'Check for supplementary mat...,[],"[{'code': 'DEMOSCI', 'encumbrance': '4a53cb5d-...",f94bd3f9-01e9-4d1f-a377-36c1227b1168,False,[{'holdingId': '61cf8fdd-4574-4d38-afd5-87d8e1...,[ecc6b76e-4b6a-49de-8378-d7edd8ba3e20],Physical Resource,Ongoing,"{'createInventory': 'Instance, Holding, Item',...",22246-1,,Springer Science+Business Media ; Springer US],bd4c36a7-548e-4278-9530-d404d7c79e46,Ongoing,[],False,User,Journal of prevention.,"{'instructions': '', 'vendorAccount': '97855',...",{'createdDate': '2026-08-02T15:27:44.004+00:00...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,046c74a6-6c38-45a3-9235-dd735c3654a4,,True,28f6e038-52d2-4a04-a14b-8dffedd1d8bb,False,[],[],False,30.0,False,[{'contributor': 'Society for International De...,"{'listUnitPrice': 1362.0, 'currency': 'USD', '...","{'isAcknowledged': False, 'isBinderyActive': T...",[],"[{'code': 'DEMOBUS', 'encumbrance': '0dcf042e-...",514cb527-60f0-4028-b605-e5f39737f544,False,[{'holdingId': '22665aec-9210-451b-a6e4-2c53a6...,[ecc6b76e-4b6a-49de-8378-d7edd8ba3e20],Physical Resource,Ongoing,"{'createInventory': 'Instance, Holding, Item',...",22216-1,[©1978]-,[Society for International Development],75b52c31-b14a-4aa1-90fd-5f333f745fe8,Ongoing,[],False,User,Development = Développement = Desarrollo.,"{'instructions': '', 'vendorAccount': '97855',...",{'createdDate': '2026-06-16T20:13:22.755+00:00...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,a0ef81e7-feae-4ad9-a478-b387a1265e46,NaN,True,28f6e038-52d2-4a04-a14b-8dffedd1d8bb,False,[],[],True,30.0,False,[],"{'listUnitPrice': 100.0, 'currency': 'USD', 'd...","{'receivingNote': 'check for loose pieces', 'i...",[],"[{'code': 'SERIALLAW', 'encumbrance': '70a1f4b...",e94b6fcc-5e53-4084-ad4c-44087842b217,False,[{'holdingId': 'f9a6410a-7481-44ec-a10f-175f4b...,[e3775cef-9d30-4896-a452-98a099cf9bfa],Physical Resource,Ongoing,"{'createInventory': 'Instance, Holding, Item',...",21712-1,NaN,NaN,f74c9128-0666-43c9-8698-0eeea8dfc036,Ongoing,[],False,User,Journal of Aging,"{'instructions': '', 'vendorAccount': '', 'ref...",{'createdDate': '2025-05-29T12:00:20.116+00:00...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,ac6455ff-c25a-4131-aa72-69ad98d2142c,NaN,False,28f6e038-52d2-4a04-a14b-8dffedd1d8bb,False,[],[],False,5.0,False,[],"{'listUnitPriceElectronic': 1000.0, 'currency'...","{'isAcknowledged': False, 'isBinderyActive': F...",[],"[{'code': 'DEMOBUS', 'encumbrance': '4ffe53e8-...",33e9764c-c8c8-49fd-87b0-cd363f169e74,False,[{'locationId': 'd0aecd46-fc34-47c2-8702-660b6...,[d0aecd46-fc34-47c2-8702-660b6e904656],Electronic Resource,Ongoing,NaN,22069-1,NaN,NaN,905a3a93-4336-491a-80b8-7fa182caa837,Ongoing,[],False,User,Lexis Nexis Database,"{'instructions': '', 'vendorAccount': '', 'ref...",{'createdDate': '2026-03-05T20:40:29.113+00:00...,"{'activated': False, 'createInventory': 'Insta...",NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,3fa1932c-1250-4851-90cc-fea9424b7a18,NaN,True,28f6e038-52d2-4a04-a14b-8dffedd1d8bb,False,[],[],True,1.0,False,"[{'contributor': 'The Teaching Company', 'cont...","{'listUnitPrice': 21.0, 'currency': 'USD', 'di...","{'receivingNote': '12 issues per year, plus a ...",[],[],84d8a248-60c5-41e7-8b64-f9f2c8b8e981,False,[{'holdingId': 'a2a84370-0b0b-412a-a5be-a831ed...,[ecc6b76e-4b6a-49de-8378-d7edd8ba3e20],Physical Resource,Ongoing,"{'createInventory': 'Instance, Holding, Item'

In [52]:
vendors_raw = fetch_all_records(
    "/organizations-storage/organizations",
    records_key="organizations",
    query='isVendor = True'
    ) 
vendors_df = pd.DataFrame(vendors_raw)

print(f"{len(vendors_raw)} vendors found")
vendors_df.head()

43 vendors found


,id,name,code,exportToAccounting,status,organizationTypes,aliases,addresses,phoneNumbers,emails,urls,contacts,privilegedContacts,agreements,erpCode,paymentMethod,vendorCurrencies,subscriptionInterval,edi,interfaces,accounts,isVendor,isDonor,changelogs,acqUnitIds,metadata,language,claimingInterval,expectedReceiptInterval,discountPercent,tags,description,expectedInvoiceInterval,renewalActivationInterval,expectedActivationInterval,taxId,taxPercentage
0,1dc0f2dc-91b1-41bb-bb7f-145815385366,The Teaching Company,TEACHINGCO,True,Active,[],[],[],[],[],"[{'value': 'https://www.thegreatcourses.com/',...",[],[],[],852-968,Credit Card,[USD],365.0,"{'vendorEdiType': '31B/US-SAN', 'libEdiType': ...",[],[],True,False,[],[],{'createdDate': '2022-10-26T16:07:22.584+00:00...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,d1d52da2-0f71-4364-818c-9988bad5024d,Elsevier Science,ELSEVIER,False,Active,[],[],[],[],[],[],[],[],[],NaN,NaN,[],NaN,"{'vendorEdiType': '31B/US-SAN', 'libEdiType': ...",[],[],True,False,[],[],{'createdDate': '2022-04-12T15:55:28.149+00:00...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1c4867bb-9943-4e7d-a28c-c6ae48ce7b94,Minha Biblioteca,MBIB,False,Active,[],[],[],[],[],[],[],[],[],NaN,NaN,[],NaN,"{'vendorEdiType': '31B/US-SAN', 'libEdiType': ...",[1d9b2e78-c928-4c80-8e69-afa224c46466],[],True,False,[],[],{'createdDate': '2023-03-02T19:24:43.205+00:00...,por,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,bffa2517-f2e9-4eea-b096-d8a6d6c4241a,Amazon,AMAZON,True,Active,"[22b43a15-8a41-4261-bc8e-a1dd36979d8b, 0032e4a...",[],[],[],[],[],[],[],"[{'name': 'Term1', 'discount': 2.5}]",458,Credit Card,[USD],NaN,"{'vendorEdiType': '31B/US-SAN', 'libEdiType': ...",[],"[{'name': 'Prime', 'accountNo': 'PR1357246', '...",True,False,[],[],{'createdDate': '2022-06-10T15:14:28.976+00:00...,NaN,3.0,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,b5554b37-23d5-473a-bca9-8efe19eaad6f,Emerald Publishing Limited,EPL,True,Active,"[22b43a15-8a41-4261-bc8e-a1dd36979d8b, 81829b9...",[{'value': 'EM'}],"[{'addressLine1': '21 Wellington St', 'city': ...","[{'phoneNumber': '+441133231381', 'categories'...","[{'value': 'orders@emeraldpublishing.com', 'is...",[],"[08ca5ca7-d2dc-43e8-ada0-ea0ec2a54cb3, a51ee9d...",[],[],6985302,Credit Card,[],NaN,"{'vendorEdiType': '31B/US-SAN', 'libEdiType': ...",[a33b0d24-dba1-498a-b440-011344b9841d],"[{'name': 'Electronic materials', 'accountNo':...",True,False,[],[fbc675ce-1d2a-4cdf-accc-7f977a345600],{'createdDate': '2023-04-11T18:47:06.140+00:00...,NaN,NaN,NaN,2.0,{'tagList': []},NaN,NaN,NaN,NaN,NaN,NaN


In [46]:
period_start="2026-07-01"
period_end="2027-06-30"

In [47]:
invoices_raw = fetch_all_records(
    "/invoice-storage/invoices",
    records_key="invoices",
    query='invoiceDate >= ' + period_start + ' and invoiceDate <= ' + period_end
    ) 
invoices_df = pd.DataFrame(invoices_raw)

print(f"{len(invoices_raw)} invoices found between {period_start} and {period_end}")
invoices_df.head()

3 invoices found between 2026-07-01 and 2027-06-30


,id,accountingCode,adjustments,adjustmentsTotal,batchGroupId,chkSubscriptionOverlap,currency,enclosureNeeded,exportToAccounting,folioInvoiceNo,invoiceDate,paymentDue,paymentMethod,status,source,subTotal,total,vendorInvoiceNo,poNumbers,vendorId,fiscalYearId,acqUnitIds,nextInvoiceLineNumber,metadata,approvedBy,approvalDate,exchangeRate,paymentDate,voucherNumber
0,aa3a21b8-6b02-4b80-bba2-5c5d6af85738,114220706,[],0.0,2624d233-a969-49cf-9ad9-b68515cde55a,True,USD,False,False,19672,2026-07-28T00:00:00.000+00:00,2026-08-28T00:00:00.000+00:00,Deposit Account,Open,User,45000.00,45000.00,ebsco07282026,[22233],88b463bc-db54-4144-9817-7f6ee6c88372,d01d3d59-a8e7-4b58-a485-5146f5747dbb,[],3,{'createdDate': '2026-07-28T16:37:31.543+00:00...,NaN,NaN,NaN,NaN,NaN
1,d18fc07c-3171-48bd-8421-4e326faa51d5,98741,[],0.0,2a2cb998-1437-41d1-88ad-01930aaeadd5,True,USD,False,True,19606,2026-07-22T00:00:00.000+00:00,NaN,Physical Check,Paid,User,26.00,26.00,665885,[22228],e54ed4ea-943d-48d7-a194-da530bbdd9eb,d01d3d59-a8e7-4b58-a485-5146f5747dbb,[],2,{'createdDate': '2026-07-22T15:17:44.177+00:00...,b898c62b-4080-45d5-aae7-63b8c52a6de3,2026-07-22T15:17:55.545+00:00,1.0,2026-07-22T15:17:58.072+00:00,lib5658
2,735e77e2-8504-4bfd-93e2-3ea1c4411604,98741,[],0.0,2a2cb998-1437-41d1-88ad-01930aaeadd5,True,USD,False,True,19639,2026-07-22T00:00:00.000+00:00,NaN,Physical Check,Paid,User,24.95,24.95,8899874,[22230],e54ed4ea-943d-48d7-a194-da530bbdd9eb,d01d3d59-a8e7-4b58-a485-5146f5747dbb,[],2,{'createdDate': '2026-07-22T20:33:09.702+00:00...,b898c62b-4080-45d5-aae7-63b8c52a6de3,2026-07-22T20:33:44.956+00:00,1.0,2026-07-22T20:33:46.604+00:00,lib5691


In [48]:
invoiceLines_raw = fetch_all_records(
    "/invoice-storage/invoice-lines",
    records_key="invoiceLines",
    query='metadata.createdDate >= ' + period_start + ' and metadata.createdDate <= ' + period_end
    ) 
invoiceLines_df = pd.DataFrame(invoiceLines_raw)

print(f"{len(invoiceLines_raw)} invoices found between {period_start} and {period_end}")
invoiceLines_df.head()

25 invoices found between 2026-07-01 and 2027-06-30


,id,adjustments,adjustmentsTotal,description,fundDistributions,invoiceId,invoiceLineNumber,invoiceLineStatus,quantity,releaseEncumbrance,subTotal,total,referenceNumbers,metadata,accountingCode,accountNumber,poLineId
0,6e021fed-0b27-4415-bbc9-65a1fd9b29fc,[],0.0,ANCIENT SOUTHWESTERN MORTUARY PRACTICES,[],7ce22df1-15c8-4bbb-b87a-253ca70f31b3,1,Open,1,True,99.0,99.0,[],{'createdDate': '2026-08-05T19:23:22.210+00:00...,NaN,NaN,NaN
1,799a1220-a93b-494a-89d8-e529d9023ec7,[],0.0,"ATTENTIONAL SELECTION: TOP-DOWN, BOTTOM-UP AND...",[],d638cc18-6185-410e-b6c1-3c00a2154996,1,Open,1,True,115.0,115.0,[],{'createdDate': '2026-08-05T19:23:22.324+00:00...,NaN,NaN,NaN
2,64e31419-e797-4502-8ca3-a60549e39d87,[],0.0,CROSS-BORDER INTERBANK CONTAGION RISK ANALYSIS...,[],d638cc18-6185-410e-b6c1-3c00a2154996,2,Open,1,True,115.0,115.0,[],{'createdDate': '2026-08-05T19:23:22.399+00:00...,NaN,NaN,NaN
3,46ffa61a-a2da-4cc8-adb1-eb070994ab39,[],0.0,MONOTHEISM AND RELIGIOUS DIVERSITY.,[],d638cc18-6185-410e-b6c1-3c00a2154996,4,Open,1,True,115.0,115.0,[],{'createdDate': '2026-08-05T19:23:22.523+00:00...,NaN,NaN,NaN
4,9235ada2-4b09-4cc8-8720-480aba85ff14,[],0.0,OFFERING THEORY: READING IN SOCIOGRAPHY.,[],d638cc18-6185-410e-b6c1-3c00a2154996,5,Open,1,True,140.0,140.0,[],{'createdDate': '2026-08-05T19:23:22.587+00:00...,NaN,NaN,NaN


In [53]:
# Step 1: merge POs to vendors
orders_vendors = orders_df.merge(
    vendors_df,
    left_on='vendor',
    right_on='id',
    how='left',
    suffixes=('_order', '_vendor'),
)
orders_vendors.head()

,id_order,approved,approvedById,approvalDate,billTo,dateOrdered,manualPo,notes,poNumber,orderType,reEncumber,ongoing,shipTo,template,vendor,workflowStatus,acqUnitIds_order,tags_order,metadata_order,customFields,assignedTo,id_vendor,name,code,exportToAccounting,status,organizationTypes,aliases,addresses,phoneNumbers,emails,urls,contacts,privilegedContacts,agreements,erpCode,paymentMethod,vendorCurrencies,subscriptionInterval,edi,interfaces,accounts,isVendor,isDonor,changelogs,acqUnitIds_vendor,metadata_vendor,language,claimingInterval,expectedReceiptInterval,discountPercent,tags_vendor,description,expectedInvoiceInterval,renewalActivationInterval,expectedActivationInterval,taxId,taxPercentage
0,40aaa902-bfc8-4f20-af9f-e1cd729acde9,False,4a390634-f254-4777-8448-6cd7a5ad29e3,2025-04-21T17:04:42.479+00:00,789ab62a-9efa-4c70-a6ef-a3615fe4b344,2025-04-21T17:04:42.479+00:00,False,[],21624,Ongoing,True,"{'interval': 365, 'isSubscription': True, 'man...",4d098cbe-04cd-4332-9fad-24208fae34c4,74b9551e-f7e5-43e8-887f-7d6b075e9b19,ae62585b-5585-41a7-9519-5b16c1d32d54,Open,[a3f803ba-587c-4aac-b2fc-35226659c4fe],{'tagList': []},{'createdDate': '2025-04-21T16:59:34.873+00:00...,NaN,NaN,ae62585b-5585-41a7-9519-5b16c1d32d54,Sage,SAGE,False,Active,"[e1dcb569-d5ce-4ab0-9be4-3613d14867fb, 22b43a1...","[{'value': 'Sage Publications, Inc'}]","[{'addressLine1': '1400 L ST NW, STE 250', 'ci...",[],"[{'value': 'support@maildrop.cc', 'isPrimary':...","[{'value': 'https://us.sagepub.com/', 'descrip...","[961d4c58-96bc-4612-94b6-782455114fdd, 8c6b585...",[],[],UniExtAcc-42617,Credit Card,[],NaN,"{'vendorEdiType': '31B/US-SAN', 'libEdiType': ...",[9b816edb-965c-42b9-a0dc-7a58d5667281],"[{'name': 'Technology from Sage', 'accountNo':...",True,False,[],[],{'createdDate': '2022-04-12T16:15:34.271+00:00...,eng,NaN,NaN,2.0,NaN,"Sage is a global academic publisher of books, ...",NaN,NaN,NaN,NaN,NaN
1,180bcb30-7b70-4536-bd0b-6b6cf231ddfa,False,4a390634-f254-4777-8448-6cd7a5ad29e3,2025-05-07T04:31:04.111+00:00,789ab62a-9efa-4c70-a6ef-a3615fe4b344,2025-05-07T04:31:04.111+00:00,False,[],21655,Ongoing,False,"{'interval': 365, 'isSubscription': True, 'man...",NaN,5b0bf4ea-7971-4717-a77c-8ce7f2577685,fed3982c-1e92-4719-bb9b-42129018fd25,Open,[],{'tagList': []},{'createdDate': '2025-05-07T04:29:23.084+00:00...,NaN,NaN,fed3982c-1e92-4719-bb9b-42129018fd25,Taylor and Francis,TAF,False,Active,[22b43a15-8a41-4261-bc8e-a1dd36979d8b],[],[],[],[],[{'value': 'https://librarianresources.taylora...,"[70272728-ea48-4222-a5df-df3e1a266b1f, 727e33c...",[],[],TAF90041,NaN,[],NaN,"{'vendorEdiType': '31B/US-SAN', 'libEdiType': ...",[910e5c37-f5b8-4572-b3aa-2915e4a0211c],[],True,False,[],[],{'createdDate': '2024-06-10T22:37:15.874+00:00...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,e02be3d3-63b4-45a8-a82d-d8e001adb813,False,7c2b76f7-634e-47d1-9aa5-9910751f23c8,2025-05-09T11:10:06.741+00:00,789ab62a-9efa-4c70-a6ef-a3615fe4b344,2025-05-09T11:10:06.741+00:00,NaN,[],21660,Ongoing,True,"{'interval': 1, 'isSubscription': True, 'manua...",4d098cbe-04cd-4332-9fad-24208fae34c4,NaN,1dc0f2dc-91b1-41bb-bb7f-145815385366,Open,[],NaN,{'createdDate': '2025-05-09T11:06:25.857+00:00...,NaN,NaN,1dc0f2dc-91b1-41bb-bb7f-145815385366,The Teaching Company,TEACHINGCO,True,Active,[],[],[],[],[],"[{'value': 'https://www.thegreatcourses.com/',...",[],[],[],852-968,Credit Card,[USD],365.0,"{'vendorEdiType': '31B/US-SAN', 'libEdiType': ...",[],[],True,False,[],[],{'createdDate': '2022-10-26T16:07:22.584+00:00...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,f74c9128-0666-43c9-8698-0eeea8dfc036,False,91d619a5-66b5-45f3-8680-42525ec90bcd,2025-05-29T12:00:20.752+00:00,NaN,2025-05-29T12:00:20.752+00:00,False,[],21712,Ongoing,False,"{'interval': 12, 'isSubscription': True, 'manu...",NaN,79d99241-f68e-474e-a104-da82219edb9e,1ce90497-a8ba-4940-9ffc-0044793e91a0,Open,[5f3bbae0-5d37-460f-a47f-71224c2952a5],{'tagList': []},{'createdDate': '2025-05-29T11:57:43.165+00:00...,NaN,NaN,1ce90497-a8ba-4940-9ffc-0044793e91a0,Lexis

In [57]:
# Step 2: merge POs to POLs
orders_pols = orders_vendors.merge(
    poLines_df,
    left_on='id_order',
    right_on='purchaseOrderId',
    how='left',
    suffixes=('_order', '_POL'),
)
orders_pols.head()

,id_order,approved,approvedById,approvalDate,billTo,dateOrdered,manualPo,notes,poNumber,orderType,reEncumber,ongoing,shipTo,template,vendor,workflowStatus,acqUnitIds_order,tags_order,metadata_order,customFields_order,assignedTo,id_vendor,name,code,exportToAccounting,status,organizationTypes,aliases,addresses,phoneNumbers,emails,urls,contacts,privilegedContacts,agreements,erpCode,paymentMethod,vendorCurrencies,subscriptionInterval,edi,interfaces,accounts,isVendor,isDonor,changelogs,acqUnitIds_vendor,metadata_vendor,language,claimingInterval_order,expectedReceiptInterval,discountPercent,tags_vendor,description_order,expectedInvoiceInterval,renewalActivationInterval,expectedActivationInterval,taxId,taxPercentage,id,edition,checkinItems,acquisitionMethod,automaticExport,alerts,claims,claimingActive,claimingInterval_POL,collection,contributors,cost,details,donorOrganizationIds,fundDistribution,instanceId,isPackage,locations,searchLocationIds,orderFormat,paymentStatus,physical,poLineNumber,publicationDate,publisher,purchaseOrderId,receiptStatus,reportingCodes,rush,source,titleOrPackage,vendorDetail,metadata,eresource,requester,cancellationRestriction,cancellationRestrictionNote,description_POL,receiptDate,renewalNote,customFields_POL
0,40aaa902-bfc8-4f20-af9f-e1cd729acde9,False,4a390634-f254-4777-8448-6cd7a5ad29e3,2025-04-21T17:04:42.479+00:00,789ab62a-9efa-4c70-a6ef-a3615fe4b344,2025-04-21T17:04:42.479+00:00,False,[],21624,Ongoing,True,"{'interval': 365, 'isSubscription': True, 'man...",4d098cbe-04cd-4332-9fad-24208fae34c4,74b9551e-f7e5-43e8-887f-7d6b075e9b19,ae62585b-5585-41a7-9519-5b16c1d32d54,Open,[a3f803ba-587c-4aac-b2fc-35226659c4fe],{'tagList': []},{'createdDate': '2025-04-21T16:59:34.873+00:00...,NaN,NaN,ae62585b-5585-41a7-9519-5b16c1d32d54,Sage,SAGE,False,Active,"[e1dcb569-d5ce-4ab0-9be4-3613d14867fb, 22b43a1...","[{'value': 'Sage Publications, Inc'}]","[{'addressLine1': '1400 L ST NW, STE 250', 'ci...",[],"[{'value': 'support@maildrop.cc', 'isPrimary':...","[{'value': 'https://us.sagepub.com/', 'descrip...","[961d4c58-96bc-4612-94b6-782455114fdd, 8c6b585...",[],[],UniExtAcc-42617,Credit Card,[],NaN,"{'vendorEdiType': '31B/US-SAN', 'libEdiType': ...",[9b816edb-965c-42b9-a0dc-7a58d5667281],"[{'name': 'Technology from Sage', 'accountNo':...",True,False,[],[],{'createdDate': '2022-04-12T16:15:34.271+00:00...,eng,NaN,NaN,2.0,NaN,"Sage is a global academic publisher of books, ...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,180bcb30-7b70-4536-bd0b-6b6cf231ddfa,False,4a390634-f254-4777-8448-6cd7a5ad29e3,2025-05-07T04:31:04.111+00:00,789ab62a-9efa-4c70-a6ef-a3615fe4b344,2025-05-07T04:31:04.111+00:00,False,[],21655,Ongoing,False,"{'interval': 365, 'isSubscription': True, 'man...",NaN,5b0bf4ea-7971-4717-a77c-8ce7f2577685,fed3982c-1e92-4719-bb9b-42129018fd25,Open,[],{'tagList': []},{'createdDate': '2025-05-07T04:29:23.084+00:00...,NaN,NaN,fed3982c-1e92-4719-bb9b-42129018fd25,Taylor and Francis,TAF,False,Active,[22b43a15-8a41-4261-bc8e-a1dd36979d8b],[],[],[],[],[{'value': 'https://librarianresources.taylora...,"[70272728-ea48-4222-a5df-df3e1a266b1f, 727e33c...",[],[],TAF90041,NaN,[],NaN,"{'vendorEdiType': '31B/US-SAN', 'libEdiType': ...",[910e5c37-f5b8-4572-b3aa-2915e4a0211c],[],True,False,[],[],{'createdDate': '2024-06-10T22:37:15.874+00:00...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,b963433a-3305-480d-a29e-ab177a3587b6,NaN,True,28f6e038-52d2-4a04-a14b-8dffedd1d8bb,False,[],[],False,NaN,False,[],"{'listUnitPriceElectronic': 340.0, 'currency':...","{'isAcknowledged': False, 'isBinderyActive': F...",[],"[{'code': 'DEMOEDU', 'encumbrance': 'fc2019f7-...",NaN,False,[],[],Electronic Resource,Ongoing,NaN,21655-1,NaN,Taylor and Francis,180bcb30-7b70-4536-bd0b-6b6cf231ddfa,Ongoing,[],False,User,Distance Education,"{'instructions': '', 'vendorAccount': 'tf1234'...",{'createdDate': '2025-05-07T04:31:0

In [55]:
# Step 2: merge invoice lines to invoices
invoices_lines = invoices_df.merge(
    invoiceLines_df,
    left_on='id',
    right_on='invoiceId',
    how='left',
    suffixes=('_invoice', '_invoice-line'),
)
invoices_lines.head()

,id_invoice,accountingCode_invoice,adjustments_invoice,adjustmentsTotal_invoice,batchGroupId,chkSubscriptionOverlap,currency,enclosureNeeded,exportToAccounting,folioInvoiceNo,invoiceDate,paymentDue,paymentMethod,status,source,subTotal_invoice,total_invoice,vendorInvoiceNo,poNumbers,vendorId,fiscalYearId,acqUnitIds,nextInvoiceLineNumber,metadata_invoice,approvedBy,approvalDate,exchangeRate,paymentDate,voucherNumber,id_invoice-line,adjustments_invoice-line,adjustmentsTotal_invoice-line,description,fundDistributions,invoiceId,invoiceLineNumber,invoiceLineStatus,quantity,releaseEncumbrance,subTotal_invoice-line,total_invoice-line,referenceNumbers,metadata_invoice-line,accountingCode_invoice-line,accountNumber,poLineId
0,aa3a21b8-6b02-4b80-bba2-5c5d6af85738,114220706,[],0.0,2624d233-a969-49cf-9ad9-b68515cde55a,True,USD,False,False,19672,2026-07-28T00:00:00.000+00:00,2026-08-28T00:00:00.000+00:00,Deposit Account,Open,User,45000.00,45000.00,ebsco07282026,[22233],88b463bc-db54-4144-9817-7f6ee6c88372,d01d3d59-a8e7-4b58-a485-5146f5747dbb,[],3,{'createdDate': '2026-07-28T16:37:31.543+00:00...,NaN,NaN,NaN,NaN,NaN,0ea7e6fe-108c-4435-8371-e0a001ba55d7,[],0.0,Academic Search Ultimate,"[{'code': 'DEMOEDU', 'fundId': '880e5be1-0c49-...",aa3a21b8-6b02-4b80-bba2-5c5d6af85738,1,Open,1,True,22500.00,22500.00,[],{'createdDate': '2026-07-28T16:39:37.522+00:00...,114220706,97855,c8465a0f-26af-4749-bf66-5f6f88702234
1,aa3a21b8-6b02-4b80-bba2-5c5d6af85738,114220706,[],0.0,2624d233-a969-49cf-9ad9-b68515cde55a,True,USD,False,False,19672,2026-07-28T00:00:00.000+00:00,2026-08-28T00:00:00.000+00:00,Deposit Account,Open,User,45000.00,45000.00,ebsco07282026,[22233],88b463bc-db54-4144-9817-7f6ee6c88372,d01d3d59-a8e7-4b58-a485-5146f5747dbb,[],3,{'createdDate': '2026-07-28T16:37:31.543+00:00...,NaN,NaN,NaN,NaN,NaN,6badef28-b2c6-46bb-a3d7-0025e99f9eab,[],0.0,Business Source Ultimate,"[{'code': 'DEMOBUS', 'fundId': '33d2a1e4-8392-...",aa3a21b8-6b02-4b80-bba2-5c5d6af85738,2,Open,1,True,22500.00,22500.00,[],{'createdDate': '2026-07-28T16:39:55.838+00:00...,114220706,97855,d7c7eea3-0270-48ac-8241-f8534b78f0b2
2,d18fc07c-3171-48bd-8421-4e326faa51d5,98741,[],0.0,2a2cb998-1437-41d1-88ad-01930aaeadd5,True,USD,False,True,19606,2026-07-22T00:00:00.000+00:00,NaN,Physical Check,Paid,User,26.00,26.00,665885,[22228],e54ed4ea-943d-48d7-a194-da530bbdd9eb,d01d3d59-a8e7-4b58-a485-5146f5747dbb,[],2,{'createdDate': '2026-07-22T15:17:44.177+00:00...,b898c62b-4080-45d5-aae7-63b8c52a6de3,2026-07-22T15:17:55.545+00:00,1.0,2026-07-22T15:17:58.072+00:00,lib5658,c5a1f627-e03e-4b79-9086-ee72e40f8a6d,[],0.0,GEOLOGY UNDERFOOT ON COLORADO'S WESTERN SLOPE.,"[{'code': 'DEMOSCI', 'encumbrance': 'bdd9ca56-...",d18fc07c-3171-48bd-8421-4e326faa51d5,1,Paid,1,True,26.00,26.00,"[{'refNumber': '90104687238', 'refNumberType':...",{'createdDate': '2026-07-22T15:17:45.930+00:00...,98741,891010,577aac3d-9df8-4953-9e79-b6c96d8c07ce
3,735e77e2-8504-4bfd-93e2-3ea1c4411604,98741,[],0.0,2a2cb998-1437-41d1-88ad-01930aaeadd5,True,USD,False,True,19639,2026-07-22T00:00:00.000+00:00,NaN,Physical Check,Paid,User,24.95,24.95,8899874,[22230],e54ed4ea-943d-48d7-a194-da530bbdd9eb,d01d3d59-a8e7-4b58-a485-5146f5747dbb,[],2,{'createdDate': '2026-07-22T20:33:09.702+00:00...,b898c62b-4080-45d5-aae7-63b8c52a6de3,2026-07-22T20:33:44.956+00:00,1.0,2026-07-22T20:33:46.604+00:00,lib5691,e46aa656-c884-4bd6-9fd0-67f928584b31,[],0.0,HIKING COLORADO'S WESTERN SLOPE.,"[{'code': 'DEMOSCI', 'encumbrance': '6944f313-...",735e77e2-8504-4bfd-93e2-3ea1c4411604,1,Paid,1,True,24.95,24.95,"[{'refNumber': '90104690616', 'refNumberType':...",{'createdDate': '2026-07-22T20:33:11.076+00:00...,98741,891010,79df8804-c90e-4463-a4ad-518e64ce17ab


In [56]:
# Step 2: merge orders and invoices
orders_full = orders_pols.merge(
    invoices_lines,
    left_on='id_POL',
    right_on='poLineId',
    how='left',
    suffixes=('_ord', '_inv'),
)
orders_full.head()

KeyError: 'id_POL'